# `denovo.run` — Self-Hosted WebGPU Backend on Google Colab (Experimental)

Run all 21 WebGPU models in **ProteinDesignSpace (`denovo.run`)** on a Colab **T4 / L4 / A100** GPU with **zero Python ML dependencies** (headless Chromium + Vulkan WebGPU + Cloudflare Quick Tunnel + `HttpOnly` cookie auth).

> **Before running:** Go to **Runtime → Change runtime type → Hardware accelerator: GPU** (T4, L4, or A100).
>
> **How it works:**
> 1. **Cell 1** enables the NVIDIA Vulkan ICD inside Colab and verifies `vulkaninfo --summary` sees the real GPU (not `llvmpipe`).
> 2. **Cell 2** installs Node 22, Chromium (Playwright), `cloudflared`, and builds the `denovo.run` web bundle (~60–90s).
> 3. **Cell 3** starts `server/selfhost-server.mjs` + a private Cloudflare Quick Tunnel and prints your **`https://*.trycloudflare.com`** URL and one-time **API Token**.

In [ ]:
#@title 1. Check GPU & Enable NVIDIA Vulkan ICD
import json, os, shutil, subprocess

def sh(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=False)

gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    text=True, capture_output=True,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected! Switch Runtime -> Change runtime type -> T4/L4/A100 GPU first.')

gpu_line = gpu_probe.stdout.strip().splitlines()[0]
driver_ver = gpu_line.split(',')[-1].strip()
driver_major = driver_ver.split('.')[0]
print(f'Detected GPU: {gpu_line}')

# Install Vulkan loader + tools
sh('apt-get update -qq && apt-get install -qq -y --no-install-recommends libvulkan1 vulkan-tools libglvnd0 libxext6')

# Locate libGLX_nvidia.so.0 (usually pre-mounted by Colab; fallback to matching libnvidia-gl if missing)
def find_nvidia_glx():
    out = subprocess.run('ldconfig -p | grep libGLX_nvidia.so.0 || true', shell=True, text=True, capture_output=True).stdout
    for line in out.splitlines():
        if '=>' in line:
            p = line.split('=>', 1)[1].strip()
            if os.path.exists(p):
                return p
    for cand in ['/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0', '/usr/lib64/libGLX_nvidia.so.0']:
        if os.path.exists(cand):
            return cand
    return None

glx_path = find_nvidia_glx()
if not glx_path:
    print(f'libGLX_nvidia.so.0 not pre-mounted; trying libnvidia-gl-{driver_major}...')
    sh(f'apt-get install -qq -y --no-install-recommends libnvidia-gl-{driver_major} || true', check=False)
    glx_path = find_nvidia_glx()

print('NVIDIA GLX/Vulkan ICD library:', glx_path or 'NOT FOUND (may fall back to llvmpipe)')

os.makedirs('/usr/share/vulkan/icd.d', exist_ok=True)
icd_path = '/usr/share/vulkan/icd.d/nvidia_icd.json'
if not os.path.exists(icd_path):
    with open(icd_path, 'w') as f:
        json.dump({
            'file_format_version': '1.0.0',
            'ICD': {
                'library_path': glx_path or 'libGLX_nvidia.so.0',
                'api_version': '1.3.0',
            },
        }, f, indent=2)
    print(f'Wrote {icd_path}')

os.environ['VK_ICD_FILENAMES'] = icd_path
vinfo = subprocess.run(['vulkaninfo', '--summary'], text=True, capture_output=True)
print(vinfo.stdout or vinfo.stderr)
if 'PHYSICAL_DEVICE_TYPE_DISCRETE_GPU' not in vinfo.stdout:
    print('WARNING: vulkaninfo did not report a discrete NVIDIA GPU. Headless Chrome may fall back to software WebGPU (SwiftShader).')
else:
    print('SUCCESS: Vulkan sees your discrete NVIDIA GPU!')

In [ ]:
#@title 2. Install Node 22, Chromium, cloudflared & Build ProteinDesignSpace
REPO_URL = "https://github.com/ramithuh/ProteinDesignSpace.git" #@param {type:"string"}
BRANCH = "feat/unifying-engine" #@param {type:"string"}

import os, subprocess

NODE_DIR = '/opt/node22'
NODE_BIN = f'{NODE_DIR}/bin'
os.environ['PATH'] = f"{NODE_BIN}:{os.environ.get('PATH', '')}"

def run(cmd):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, env=os.environ)

# 1. Standalone Node 22 (fast, avoids apt conflicts)
if not os.path.exists(f'{NODE_BIN}/node'):
    run('curl -fsSL https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz | tar -xJ -C /opt && mv /opt/node-v22.14.0-linux-x64 /opt/node22')
run('node -v && npm -v')

# 2. Clone or update repo
REPO_DIR = '/content/ProteinDesignSpace'
if not os.path.exists(REPO_DIR):
    run(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}')
else:
    run(f'git -C {REPO_DIR} fetch origin {BRANCH} --depth 1 && git -C {REPO_DIR} reset --hard origin/{BRANCH}')

# 3. Build static web app + install playwright-core & Chromium
run(f'npm --prefix {REPO_DIR}/web ci && npm --prefix {REPO_DIR}/web run build')
run(f'npm --prefix {REPO_DIR}/server install playwright-core')
run(f'npx --prefix {REPO_DIR}/server playwright-core install chromium && npx --prefix {REPO_DIR}/server playwright-core install-deps chromium')

# 4. Download cloudflared static binary for zero-signup HTTPS Quick Tunnel
if not os.path.exists('/usr/local/bin/cloudflared'):
    run('curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
print('Setup complete! Run Cell 3 to start your Host GPU server.')

In [ ]:
#@title 3. Launch denovo.run Host GPU Server + HTTPS Quick Tunnel
CUSTOM_API_TOKEN = "" #@param {type:"string"}
MAX_FOLD_RESIDUES = 2048 #@param {type:"integer"}
ALLOW_SOFTWARE_FALLBACK = False #@param {type:"boolean"}

import os, re, secrets, subprocess, time
from IPython.display import HTML, display

REPO_DIR = '/content/ProteinDesignSpace'
NODE_BIN = '/opt/node22/bin'
PORT = 8787
API_TOKEN = CUSTOM_API_TOKEN.strip() or f'dnv_{secrets.token_hex(18)}'

# Stop any prior server/tunnel in this runtime
subprocess.run('pkill -f selfhost-server.mjs || true; pkill -f "cloudflared tunnel" || true', shell=True)
time.sleep(1)

env = os.environ.copy()
env.update({
    'PATH': f"{NODE_BIN}:{env.get('PATH', '')}",
    'HOST': '127.0.0.1',
    'PORT': str(PORT),
    'DENOVO_API_TOKEN': API_TOKEN,
    'DENOVO_DATA_DIR': '/tmp/denovo-profile',
    'DENOVO_ALLOWED_ORIGINS': 'https://denovo.run',
    'DENOVO_DISABLE_SANDBOX': '1',
    'DENOVO_DISABLE_GPU_SANDBOX': '1',
    'DENOVO_MAX_FOLD_RESIDUES': str(MAX_FOLD_RESIDUES),
    'VK_ICD_FILENAMES': '/usr/share/vulkan/icd.d/nvidia_icd.json',
})
if ALLOW_SOFTWARE_FALLBACK:
    env['DENOVO_ALLOW_SOFTWARE_GPU'] = '1'

server_log_path = '/tmp/denovo-server.log'
server_log = open(server_log_path, 'w')
server_proc = subprocess.Popen(
    [f'{NODE_BIN}/node', f'{REPO_DIR}/server/selfhost-server.mjs'],
    env=env, stdout=server_log, stderr=subprocess.STDOUT,
)

# Wait up to 25s for headless Chromium + Vulkan WebGPU to initialize
ready_line = ''
for _ in range(25):
    time.sleep(1)
    with open(server_log_path) as f:
        txt = f.read()
    if 'Host GPU ready:' in txt:
        ready_line = [l for l in txt.splitlines() if 'Host GPU ready:' in l][-1]
        break
    if server_proc.poll() is not None:
        print(txt)
        raise RuntimeError('selfhost-server.mjs exited unexpectedly during startup.')

with open(server_log_path) as f:
    print(f.read())
if not ready_line:
    raise RuntimeError('Timed out waiting for Host GPU ready. Check log above.')

# Start Cloudflare Quick Tunnel
tunnel_log_path = '/tmp/denovo-tunnel.log'
tunnel_log = open(tunnel_log_path, 'w')
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = ''
for _ in range(20):
    time.sleep(1)
    with open(tunnel_log_path) as f:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', f.read())
    if m:
        public_url = m.group(0)
        break

if not public_url:
    with open(tunnel_log_path) as f:
        print(f.read())
    raise RuntimeError('Could not obtain Cloudflare Quick Tunnel URL.')

display(HTML(f'''
<div style="font-family:ui-sans-serif,system-ui;padding:14px 16px;border:1px solid #d97706;border-radius:10px;background:#fffbeb;color:#1f2937">
  <div style="font-weight:700;font-size:15px;margin-bottom:6px">denovo.run Host GPU is live on Colab</div>
  <div style="margin-bottom:6px"><b>Status:</b> <code>{ready_line}</code></div>
  <div style="margin-bottom:6px"><b>Open Workbench:</b> <a href="{public_url}" target="_blank" rel="noopener noreferrer">{public_url}</a></div>
  <div style="margin-bottom:8px"><b>One-Time API Token:</b> <code style="user-select:all;padding:2px 6px;background:#fef3c7;border-radius:4px">{API_TOKEN}</code></div>
  <div style="font-size:12px;color:#4b5563">Click the top-right <b>GPU pill</b> → switch to <b>Host GPU</b> → leave URL blank → paste the token once and click <b>Sign in</b> (saved via HttpOnly cookie).</div>
</div>
'''))

print('Keep this cell running while you use the workbench (interrupt cell to stop).')
try:
    seen = 0
    while server_proc.poll() is None:
        with open(server_log_path) as f:
            data = f.read()
        if len(data) > seen:
            print(data[seen:], end='')
            seen = len(data)
        time.sleep(2)
except KeyboardInterrupt:
    print('\nStopping server and tunnel...')
finally:
    tunnel_proc.terminate()
    server_proc.terminate()